In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [14]:
import os
from glob import glob
from sklearn.model_selection import train_test_split
import librosa
import numpy as np
import random

from transformers import AutoFeatureExtractor
from transformers import ASTForAudioClassification

In [6]:
 
DATA_ROOT = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"
NOISE_ROOT = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio"


In [7]:
recipes = []

genres = sorted(os.listdir(DATA_ROOT))
noise_files = glob(os.path.join(NOISE_ROOT, "*.wav"))

for genre in genres:
    genre_path = os.path.join(DATA_ROOT, genre)
    song_dirs = glob(os.path.join(genre_path, "*"))
    
    for song_dir in song_dirs:
        stem_paths = glob(os.path.join(song_dir, "*.wav"))
        
        if len(stem_paths) < 4:
            continue
        
        recipes.append({
            "genre": genre,
            "stems": stem_paths[:4],
            "noise": None  
        })
 
train_recipes, val_recipes = train_test_split(
    recipes,
    test_size=0.2,
    shuffle=True,
    random_state=42
)

print(len(train_recipes), len(val_recipes))

800 200


In [10]:

SR = 16000
DURATION = 10
TARGET_LEN = SR * DURATION   

def load_and_fix(path):
    y, _ = librosa.load(path, sr=SR)
    
    if len(y) < TARGET_LEN:
        y = np.pad(y, (0, TARGET_LEN - len(y)))
    else:
        y = y[:TARGET_LEN]
        
    return y

def create_mix(stem_paths, noise_files):
    stems = [load_and_fix(p) for p in stem_paths]
    mix = sum(stems)
     
    noise_path = random.choice(noise_files)
    noise = load_and_fix(noise_path)
    
    mix = mix + 0.2 * noise
    
    return mix
 
mix = create_mix(recipes[0]["stems"], noise_files)
print(mix.shape)

(160000,)


In [13]:
 
extractor = AutoFeatureExtractor.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593"
)

mix = np.ones(160000)

inputs = extractor(
    mix,
    sampling_rate=16000,
    return_tensors="pt"
)

input_values = inputs["input_values"].squeeze(0)

print(input_values.shape) 

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

torch.Size([1024, 128])


In [16]:
model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    num_labels=10,
    ignore_mismatched_sizes=True
)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(num_params)

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                        
------------------------+----------+----------------------------------------------------------------------------------------
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527]) vs model:torch.Size([10])          
classifier.dense.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527, 768]) vs model:torch.Size([10, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


86196490


In [18]:
y_test = np.array([-0.85, 0.40, 0.20, -0.10])

y_norm = y_test / (np.max(np.abs(y_test)) + 1e-9)

print(y_norm)


[-1.          0.47058823  0.23529412 -0.11764706]
